In [1]:
import os
import subprocess
from pathlib import Path
from dotenv import load_dotenv
import psycopg2
from psycopg2 import sql
import yaml 

In [2]:
import os, sys
from pathlib import Path

prefix = Path(sys.prefix)

# Make GDAL/PROJ + plugin drivers discoverable inside Jupyter on Windows
os.environ["PROJ_LIB"] = str(prefix / "Library" / "share" / "proj")
os.environ["GDAL_DATA"] = str(prefix / "Library" / "share" / "gdal")
os.environ["GDAL_DRIVER_PATH"] = str(prefix / "Library" / "lib" / "gdalplugins")
os.environ["PATH"] = str(prefix / "Library" / "bin") + os.pathsep + os.environ.get("PATH", "")

OGR2OGR = prefix / "Library" / "bin" / "ogr2ogr.exe"
OGRINFO = prefix / "Library" / "bin" / "ogrinfo.exe"

In [3]:
print("Python:", sys.executable)
print("ogr2ogr:", OGR2OGR, "exists:", OGR2OGR.exists())
print("ogrinfo:", OGRINFO, "exists:", OGRINFO.exists())

# Confirm Postgres driver exists
p = subprocess.run([str(OGRINFO), "--formats"], capture_output=True, text=True)
print("Has PostgreSQL driver:", "PostgreSQL -vector-" in p.stdout)

Python: c:\Users\Zachary\anaconda3\envs\cel-export\python.exe
ogr2ogr: c:\Users\Zachary\anaconda3\envs\cel-export\Library\bin\ogr2ogr.exe exists: True
ogrinfo: c:\Users\Zachary\anaconda3\envs\cel-export\Library\bin\ogrinfo.exe exists: True
Has PostgreSQL driver: True


In [4]:
def require_ogr2ogr():
    subprocess.run([str(OGR2OGR), "--version"], check=True, capture_output=True, text=True)

In [5]:
subprocess.run([str(OGR2OGR), "--formats"], capture_output=True, text=True).stdout

'Supported Formats: (ro:read-only, rw:read-write, +:update, v:virtual-I/O s:subdatasets)\n  FITS -raster,vector- (rw+): Flexible Image Transport System (*.fits)\n  PCIDSK -raster,vector- (rw+v): PCIDSK Database File (*.pix)\n  netCDF -raster,multidimensional raster,vector- (rw+s): Network Common Data Format (*.nc)\n  PDS4 -raster,vector- (rw+vs): NASA Planetary Data System 4 (*.xml)\n  VICAR -raster,vector- (rw+v): MIPL VICAR file\n  JP2OpenJPEG -raster,vector- (rwv): JPEG-2000 driver based on JP2OpenJPEG library (*.jp2, *.j2k)\n  PDF -raster,vector- (rw+vs): Geospatial PDF (*.pdf)\n  MBTiles -raster,vector- (rw+v): MBTiles (*.mbtiles)\n  TileDB -raster,multidimensional raster,vector- (rw+vs): TileDB\n  BAG -raster,multidimensional raster,vector- (rw+v): Bathymetry Attributed Grid (*.bag)\n  EEDA -vector- (ro): Earth Engine Data API\n  OGCAPI -raster,vector- (rov): OGCAPI\n  ESRI Shapefile -vector- (rw+v): ESRI Shapefile (*.shp, *.dbf, *.shz, *.shp.zip)\n  MapInfo File -vector- (rw+v):

In [6]:
# ----------------------------
# paths
# ----------------------------
project_root = Path.cwd().parents[0]
config_path = project_root / "configs" / "paths.yaml"

with open(config_path) as f:
    paths = yaml.safe_load(f)

data_external = project_root / paths["data"]["external"]
data_processed = project_root / paths["data"]["processed"]

OUT_DIR = data_processed / "geoparquets"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
# ----------------------------
# env / db connection
# ----------------------------
load_dotenv()

PGDATABASE = os.getenv("PGDATABASE")
PGUSER     = os.getenv("PGUSER")
PGPASSWORD = os.getenv("PGPASSWORD")
PGHOST     = os.getenv("PGHOST")
PGPORT     = os.getenv("PGPORT")

if not all([PGDATABASE, PGUSER, PGPASSWORD, PGHOST, PGPORT]):
    missing = [k for k,v in {
        "PGDATABASE": PGDATABASE,
        "PGUSER": PGUSER,
        "PGPASSWORD": PGPASSWORD,
        "PGHOST": PGHOST,
        "PGPORT": PGPORT
    }.items() if not v]
    raise RuntimeError(f"Missing env vars: {missing}")

conn = psycopg2.connect(
    dbname=PGDATABASE,
    user=PGUSER,
    password=PGPASSWORD,
    host=PGHOST,
    port=PGPORT,
)
conn.autocommit = False

# psycopg2 key=val conn string (not used by ogr2ogr)
CONN_STR = f"dbname={PGDATABASE} user={PGUSER} password={PGPASSWORD} host={PGHOST} port={PGPORT}"

# GDAL ogr2ogr conn string
CONN_STR_URI = (
    f"PG:dbname={PGDATABASE} user={PGUSER} password={PGPASSWORD} "
    f"host={PGHOST} port={PGPORT}"
)

SCHEMA = "public"
EXPORT_SCHEMA = "exports"   # recommended (avoid overwriting public)

In [8]:
# ----------------------------
# SQL helpers
# ----------------------------
def run_sql(q, params=None):
    with conn.cursor() as cur:
        cur.execute(q, params)

def fetchall_sql(q, params=None):
    with conn.cursor() as cur:
        cur.execute(q, params)
        return cur.fetchall()

def ensure_schema(schema_name: str):
    run_sql(sql.SQL("CREATE SCHEMA IF NOT EXISTS {}").format(sql.Identifier(schema_name)))

def relation_exists(schema: str, name: str) -> bool:
    q = """
    SELECT EXISTS (
      SELECT 1
      FROM information_schema.tables
      WHERE table_schema = %s AND table_name = %s
      UNION ALL
      SELECT 1
      FROM information_schema.views
      WHERE table_schema = %s AND table_name = %s
    );
    """
    return fetchall_sql(q, (schema, name, schema, name))[0][0]

def get_table_columns(schema: str, table: str):
    q = """
    SELECT column_name, data_type, udt_name
    FROM information_schema.columns
    WHERE table_schema = %s AND table_name = %s;
    """
    return fetchall_sql(q, (schema, table))

In [9]:
# ----------------------------
# Expression builders (SAFE for mixed-case columns)
# ----------------------------
def expr_float(col: str) -> sql.SQL:
    # NULLIF("<col>", '')::double precision
    return sql.SQL("NULLIF({}, '')::double precision").format(sql.Identifier(col))

def expr_astext(col: str) -> sql.SQL:
    return sql.SQL("ST_AsText({})").format(sql.Identifier(col))

In [10]:
# ----------------------------
# Standardization config
# ----------------------------
RENAME_TABLES = {
    "joined_grid_15k_v3":   "cel_grid_15k",
    "joined_grid_62k_v3":   "cel_grid_62k",
    "joined_grid_250k_v3":  "cel_grid_250k",
    "joined_hex_15k":       "cel_hex_15k",
    "joined_hex_62k":       "cel_hex_62k",
    "joined_hex_250k":      "cel_hex_250k",
}

GRID_15K = {"joined_grid_15k_v3"}
GRID_BIG = {"joined_grid_62k_v3", "joined_grid_250k_v3"}
HEX_15K  = {"joined_hex_15k"}
HEX_BIG  = {"joined_hex_62k", "joined_hex_250k"}

BASE_COMMON_GRID = {
    "oid": "OID",
    "adm2_en": "ADM2_EN",
    "adm1_en": "ADM1_EN",
    "adm0_en": "ADM0_EN",
    "name": "name",
    "lat": "lat",
    "lon": "lon",
}

BASE_COMMON_HEX = {
    "oid": "OID",
    "adm2_en": "ADM2_EN",
    "adm1_en": "ADM1_EN",
    "adm0_en": "ADM0_EN",
    "name": "settlement_name",
    "lat": "lat",
    "lon": "lon",
}

DIST_COMMON_GRID = {
    "dist_to_road": "dist_to_road",
    "dist_to_market": "dist_to_market",
    "dist_to_rivers": "dist_to_rivers",
    "dist_to_rivers_and_streams": "dist_to_rivers_and_streams",
    "dist_to_rivers_plus": "dist_to_rivers_plus",
    "dist_to_pop_center_1": "dist_to_pop_center_1",
    "dist_to_pop_center_2": "dist_to_pop_center_2",
    "dist_to_pop_center_3": "dist_to_pop_center_3",

    "dist_to_road_geodesic": "dist_to_road_geodesic",
    "dist_to_market_geodesic": "dist_to_market_geodesic",
    "dist_to_rivers_geodesic": "dist_to_rivers_geodesic",
    "dist_to_rivers_and_streams_geodesic": "dist_to_rivers_and_streams_geodesic",
    "dist_to_rivers_plus_geodesic": "dist_to_rivers_plus_geodesic",
    "dist_to_pop_center_1_geodesic": "dist_to_pop_center_1_geodesic",
    "dist_to_pop_center_2_geodesic": "dist_to_pop_center_2_geodesic",
    "dist_to_pop_center_3_geodesic": "dist_to_pop_center_3_geodesic",

    "dist_to_road_diff": "dist_to_road_diff",
    "dist_to_market_diff": "dist_to_market_diff",
    "dist_to_rivers_diff": "dist_to_rivers_diff",
    "dist_to_rivers_and_streams_diff": "dist_to_rivers_and_streams_diff",
    "dist_to_rivers_plus_diff": "dist_to_rivers_plus_diff",
    "dist_to_pop_center_1_diff": "dist_to_pop_center_1_diff",
    "dist_to_pop_center_2_diff": "dist_to_pop_center_2_diff",
    "dist_to_pop_center_3_diff": "dist_to_pop_center_3_diff",

    # populated for ALL cells: distance to the nearest settlement boundary
    # (for inside cells this is the overlapping settlement; for outside cells
    # it is the nearest settlement polygon edge)
    "dist_to_settlement_boundary": "dist_to_settlement_boundary",
    "dist_to_settlement_centroid": "dist_to_settlement_centroid",

    # populated only for cells OUTSIDE every settlement boundary
    "closest_settlement": "closest_settlement",
}

DIST_COMMON_HEX = {
    "dist_to_road": "dist_to_road",
    "dist_to_market": "dist_to_market",
    "dist_to_rivers": "dist_to_rivers",
    "dist_to_rivers_and_streams": "dist_to_rivers_and_streams",
    "dist_to_rivers_plus": "dist_to_rivers_plus",
    "dist_to_pop_center_1": "dist_to_pop_center_1",
    "dist_to_pop_center_2": "dist_to_pop_center_2",
    "dist_to_pop_center_3": "dist_to_pop_center_3",
    "dist_to_pop_center_4": "dist_to_pop_center_4",

    # populated for ALL cells: distance to the nearest settlement boundary
    # (for inside cells this is the overlapping settlement; for outside cells
    # it is the nearest settlement polygon edge)
    "dist_to_settlement_boundary": "dist_to_settlement_boundary",
    "dist_to_settlement_centroid": "dist_to_settlement_centroid",

    # populated only for cells OUTSIDE every settlement boundary
    "closest_settlement": "closest_settlement",
}

CANONICAL_GRID_15K = {
    **BASE_COMMON_GRID,
    **DIST_COMMON_GRID,

    # 15k joined tables store these as TEXT (old pipeline) — use expr_float to cast
    "elev_avg":  expr_float("dem_ELEV_AVG"),
    "elev_sd":   expr_float("dem_ELEV_SD"),
    "slope_avg": expr_float("dem_SLOPE_AVG"),

    "ag_total_area": expr_float("ag_total_area"),
    "ag_y_2017_pct": expr_float("ag_AG_Y_2017_PCT"),
    "ag_y_2018_pct": expr_float("ag_AG_Y_2018_PCT"),
    "ag_y_2019_pct": expr_float("ag_AG_Y_2019_PCT"),
    "ag_y_2020_pct": expr_float("ag_AG_Y_2020_PCT"),
    "ag_y_2021_pct": expr_float("ag_AG_Y_2021_PCT"),
    "ag_y_2022_pct": expr_float("ag_AG_Y_2022_PCT"),
    "ag_y_2023_pct": expr_float("ag_AG_Y_2023_PCT"),
    "ag_y_2024_pct": expr_float("ag_AG_Y_2024_PCT"),

    "centroid_wkt": expr_astext("centroid"),
    "geom": None,
}

# 250k/62k joined tables store csv_* columns as float8 (new pipeline casts at CTAS time)
# so use direct string references, NOT expr_float() which would apply NULLIF(..., '')
# to an already-float8 column and fail with a type mismatch.
CANONICAL_GRID_BIG = {
    **BASE_COMMON_GRID,
    **DIST_COMMON_GRID,

    "elev_avg":  "csv_ELEV_AVG",
    "elev_sd":   "csv_ELEV_SD",
    "slope_avg": "csv_SLOPE_AVG",

    "ag_total_area": "csv_total_area",
    "ag_y_2017_pct": "csv_AG_Y_2017_PCT",
    "ag_y_2018_pct": "csv_AG_Y_2018_PCT",
    "ag_y_2019_pct": "csv_AG_Y_2019_PCT",
    "ag_y_2020_pct": "csv_AG_Y_2020_PCT",
    "ag_y_2021_pct": "csv_AG_Y_2021_PCT",
    "ag_y_2022_pct": "csv_AG_Y_2022_PCT",
    "ag_y_2023_pct": "csv_AG_Y_2023_PCT",
    "ag_y_2024_pct": "csv_AG_Y_2024_PCT",

    "centroid_wkt": expr_astext("centroid"),
    "geom": None,
}

CANONICAL_HEX_15K = {
    **BASE_COMMON_HEX,
    **DIST_COMMON_HEX,

    # 15k joined tables store these as TEXT (old pipeline) — use expr_float to cast
    "elev_avg":  expr_float("dem_ELEV_AVG"),
    "elev_sd":   expr_float("dem_ELEV_SD"),
    "slope_avg": expr_float("dem_SLOPE_AVG"),

    "ag_total_area": expr_float("ag_total_area"),
    "ag_y_2017_pct": expr_float("ag_AG_Y_2017_PCT"),
    "ag_y_2018_pct": expr_float("ag_AG_Y_2018_PCT"),
    "ag_y_2019_pct": expr_float("ag_AG_Y_2019_PCT"),
    "ag_y_2020_pct": expr_float("ag_AG_Y_2020_PCT"),
    "ag_y_2021_pct": expr_float("ag_AG_Y_2021_PCT"),
    "ag_y_2022_pct": expr_float("ag_AG_Y_2022_PCT"),
    "ag_y_2023_pct": expr_float("ag_AG_Y_2023_PCT"),
    "ag_y_2024_pct": expr_float("ag_AG_Y_2024_PCT"),

    "centroid_wkt": expr_astext("centroid"),
    "geom": None,
}

# 250k/62k hex joined tables: csv_* columns are float8 — direct references
CANONICAL_HEX_BIG = {
    **BASE_COMMON_HEX,
    **DIST_COMMON_HEX,

    "elev_avg":  "csv_ELEV_AVG",
    "elev_sd":   "csv_ELEV_SD",
    "slope_avg": "csv_SLOPE_AVG",

    "ag_total_area": "csv_total_area",
    "ag_y_2017_pct": "csv_AG_Y_2017_PCT",
    "ag_y_2018_pct": "csv_AG_Y_2018_PCT",
    "ag_y_2019_pct": "csv_AG_Y_2019_PCT",
    "ag_y_2020_pct": "csv_AG_Y_2020_PCT",
    "ag_y_2021_pct": "csv_AG_Y_2021_PCT",
    "ag_y_2022_pct": "csv_AG_Y_2022_PCT",
    "ag_y_2023_pct": "csv_AG_Y_2023_PCT",
    "ag_y_2024_pct": "csv_AG_Y_2024_PCT",

    "centroid_wkt": expr_astext("centroid"),
    "geom": None,
}

In [11]:
def pick_canonical_map(src_table: str) -> dict:
    if src_table in GRID_15K:
        return CANONICAL_GRID_15K
    if src_table in GRID_BIG:
        return CANONICAL_GRID_BIG
    if src_table in HEX_15K:
        return CANONICAL_HEX_15K
    if src_table in HEX_BIG:
        return CANONICAL_HEX_BIG
    raise KeyError(f"Table {src_table!r} not assigned to any family.")


# ----------------------------
# Auto-float prefix expansion for 250k tables
# ----------------------------
# For these tables, any column whose name starts with one of the listed prefixes
# is automatically cast to float8 and included in the export under its lowercase
# column name.  This avoids having to enumerate hundreds of monthly columns
# (SPEI, PPT, LST, Hansen, WC_AG) in the canonical map.
CANONICAL_AUTO_FLOAT_PREFIXES = {
    "joined_grid_250k_v3": ["spei_", "ppt_", "lst_", "hansen_", "wc_ag_"],
    "joined_hex_250k":     ["spei_", "ppt_", "lst_", "hansen_", "wc_ag_"],
}

In [12]:
# ----------------------------
# Geometry detection: prefer 'geometry' over 'centroid'
# ----------------------------
def detect_geom_column(schema: str, table: str) -> str:
    cols = get_table_columns(schema, table)  # list of (name, data_type, udt_name)
    geom_cols = [name for (name, _dtype, udt) in cols if udt == "geometry"]

    if not geom_cols:
        raise RuntimeError(f"No geometry column found in {schema}.{table}")

    if "geometry" in geom_cols:
        return "geometry"

    print(f"[WARN] No column literally named 'geometry' in {schema}.{table}. Using {geom_cols[0]!r}.")
    return geom_cols[0]

In [13]:
def safe_float_expr(col_name: str):
    """
    Clean a text-like numeric column and cast to double precision.
    Handles '', whitespace, '""', and quoted numerics.
    """
    return sql.SQL(
        "CASE "
        "WHEN BTRIM(REPLACE({0}::text, '\"', '')) = '' THEN NULL "
        "ELSE BTRIM(REPLACE({0}::text, '\"', ''))::double precision "
        "END"
    ).format(sql.Identifier(col_name))


def build_select_list(
    src_schema: str,
    src_table: str,
    canonical_map: dict,
    geom_src: str,
    auto_float_prefixes: list = None,
):
    """
    Build the SELECT column list for standardize_table.

    canonical_map entries: dest_col_name -> source (str identifier | sql.Composable | None)
    auto_float_prefixes: if provided, any column in the source table whose name starts with
                         one of these prefixes (case-insensitive) and is not already named
                         in canonical_map is auto-included as a float-compatible output.
    """
    column_meta = get_table_columns(src_schema, src_table)
    existing = {name for (name, _dtype, _udt) in column_meta}
    udt_map = {name: udt for (name, _dtype, udt) in column_meta}

    numeric_udts = {
        "int2", "int4", "int8",
        "float4", "float8",
        "numeric"
    }

    items = []

    for canon_name, source in canonical_map.items():
        if canon_name == "geom" or source is None:
            continue

        if isinstance(source, sql.Composable):
            items.append(
                sql.SQL("{} AS {}").format(source, sql.Identifier(canon_name))
            )
            continue

        if source not in existing:
            print(f"[WARN] {src_schema}.{src_table}: missing {source!r} for {canon_name!r}; skipping")
            continue

        items.append(
            sql.SQL("{} AS {}").format(sql.Identifier(source), sql.Identifier(canon_name))
        )

    if auto_float_prefixes:
        already_mapped_sources = {
            v for v in canonical_map.values() if isinstance(v, str)
        }

        for col in sorted(existing):
            col_lower = col.lower()

            if not any(col_lower.startswith(p.lower()) for p in auto_float_prefixes):
                continue
            if col in already_mapped_sources:
                continue
            if col == geom_src:
                continue

            udt = udt_map.get(col)

            if udt in numeric_udts:
                expr = sql.SQL("{}::double precision").format(sql.Identifier(col))
            else:
                expr = safe_float_expr(col)

            items.append(
                sql.SQL("{} AS {}").format(
                    expr,
                    sql.Identifier(col_lower),
                )
            )

    if geom_src not in existing:
        raise RuntimeError(f"{src_schema}.{src_table}: geom source {geom_src!r} not found")

    items.append(
        sql.SQL("{} AS {}").format(sql.Identifier(geom_src), sql.Identifier("geom"))
    )

    return items

In [14]:
# ----------------------------
# Standardize one table: DROP + CTAS
# ----------------------------
def standardize_table(
    src_schema: str,
    src_table: str,
    dst_schema: str,
    dst_table: str,
    canonical_map: dict,
    auto_float_prefixes: list = None,
) -> bool:
    """
    Standardize src_schema.src_table into dst_schema.dst_table using canonical_map.
    Returns True on success, False if the source relation (table or view) is missing (skip rather than raise).
    auto_float_prefixes: passed through to build_select_list for dynamic column expansion.
    """
    if not relation_exists(src_schema, src_table):
        print(f"[SKIP] Source relation not found: {src_schema}.{src_table}")
        return False

    geom_src = detect_geom_column(src_schema, src_table)
    select_items = build_select_list(
        src_schema, src_table, canonical_map, geom_src,
        auto_float_prefixes=auto_float_prefixes,
    )

    if not select_items:
        raise RuntimeError(f"{src_schema}.{src_table}: select list ended up empty")

    print(f"[INFO] Standardizing {src_schema}.{src_table} -> {dst_schema}.{dst_table}")

    run_sql(sql.SQL("DROP TABLE IF EXISTS {}.{};").format(
        sql.Identifier(dst_schema), sql.Identifier(dst_table)
    ))

    run_sql(sql.SQL("CREATE TABLE {}.{} AS SELECT {} FROM {}.{};").format(
        sql.Identifier(dst_schema),
        sql.Identifier(dst_table),
        sql.SQL(", ").join(select_items),
        sql.Identifier(src_schema),
        sql.Identifier(src_table),
    ))

    run_sql(sql.SQL("CREATE INDEX IF NOT EXISTS {} ON {}.{} USING GIST (geom);").format(
        sql.Identifier(f"{dst_table}_geom_gix"),
        sql.Identifier(dst_schema),
        sql.Identifier(dst_table),
    ))

    return True

In [15]:
def summarize_export_table(schema: str, table: str):
    q = sql.SQL("""
        SELECT
          COUNT(*) AS n,
          COUNT(*) FILTER (WHERE geom IS NULL) AS geom_nulls,
          MAX(ST_SRID(geom)) AS srid,
          COUNT(*) FILTER (WHERE elev_avg IS NULL) AS elev_nulls
        FROM {}.{};
    """).format(sql.Identifier(schema), sql.Identifier(table))

    return fetchall_sql(q)[0]


In [16]:
# ----------------------------
# Run standardization
# ----------------------------
ensure_schema(EXPORT_SCHEMA)

skipped = []
for src_table, dst_table in RENAME_TABLES.items():
    canonical      = pick_canonical_map(src_table)
    auto_prefixes  = CANONICAL_AUTO_FLOAT_PREFIXES.get(src_table)

    ok = standardize_table(
        SCHEMA, src_table,
        EXPORT_SCHEMA, dst_table,
        canonical,
        auto_float_prefixes=auto_prefixes,
    )
    if ok:
        conn.commit()
    else:
        skipped.append(src_table)

if skipped:
    print(f"\n[WARN] Skipped (source missing): {skipped}")
print("[DONE] Standardized tables created.")

for src, dst in RENAME_TABLES.items():
    if src not in skipped:
        print(dst, summarize_export_table(EXPORT_SCHEMA, dst))

[WARN] public.joined_grid_15k_v3: missing 'dist_to_settlement_boundary' for 'dist_to_settlement_boundary'; skipping
[WARN] public.joined_grid_15k_v3: missing 'dist_to_settlement_centroid' for 'dist_to_settlement_centroid'; skipping
[WARN] public.joined_grid_15k_v3: missing 'closest_settlement' for 'closest_settlement'; skipping
[INFO] Standardizing public.joined_grid_15k_v3 -> exports.cel_grid_15k
[WARN] public.joined_grid_62k_v3: missing 'dist_to_settlement_boundary' for 'dist_to_settlement_boundary'; skipping
[WARN] public.joined_grid_62k_v3: missing 'dist_to_settlement_centroid' for 'dist_to_settlement_centroid'; skipping
[WARN] public.joined_grid_62k_v3: missing 'closest_settlement' for 'closest_settlement'; skipping
[INFO] Standardizing public.joined_grid_62k_v3 -> exports.cel_grid_62k
[INFO] Standardizing public.joined_grid_250k_v3 -> exports.cel_grid_250k
[WARN] public.joined_hex_15k: missing 'dist_to_settlement_boundary' for 'dist_to_settlement_boundary'; skipping
[WARN] public

In [17]:
get_table_columns('exports', 'cel_grid_15k')

[('oid', 'integer', 'int4'),
 ('adm2_en', 'text', 'text'),
 ('adm1_en', 'text', 'text'),
 ('adm0_en', 'text', 'text'),
 ('name', 'text', 'text'),
 ('lat', 'double precision', 'float8'),
 ('lon', 'double precision', 'float8'),
 ('dist_to_road', 'double precision', 'float8'),
 ('dist_to_market', 'double precision', 'float8'),
 ('dist_to_rivers', 'double precision', 'float8'),
 ('dist_to_rivers_and_streams', 'double precision', 'float8'),
 ('dist_to_rivers_plus', 'double precision', 'float8'),
 ('dist_to_pop_center_1', 'double precision', 'float8'),
 ('dist_to_pop_center_2', 'double precision', 'float8'),
 ('dist_to_pop_center_3', 'double precision', 'float8'),
 ('dist_to_road_geodesic', 'double precision', 'float8'),
 ('dist_to_market_geodesic', 'double precision', 'float8'),
 ('dist_to_rivers_geodesic', 'double precision', 'float8'),
 ('dist_to_rivers_and_streams_geodesic', 'double precision', 'float8'),
 ('dist_to_rivers_plus_geodesic', 'double precision', 'float8'),
 ('dist_to_pop_cen

In [18]:
def export_table_to_geoparquet(schema: str, table: str, out_path: Path):
    # delete existing file first
    if out_path.exists():
        print(f"[INFO] Removing existing file: {out_path}")
        out_path.unlink()

    sql_query = f'SELECT * FROM "{schema}"."{table}"'

    cmd = [
        str(OGR2OGR),
        "-f", "Parquet",
        str(out_path),
        CONN_STR_URI,
        "-sql", sql_query,
        "-nln", table,
        "-geomfield", "geom",
        "-lco", "GEOMETRY_NAME=geom",
        "-lco", "COMPRESSION=ZSTD",
        "-overwrite",
    ]

    print(f"[INFO] Exporting {schema}.{table} -> {out_path}")
    proc = subprocess.run(cmd, capture_output=True, text=True)

    if proc.returncode != 0:
        print("---- ogr2ogr STDERR ----")
        print(proc.stderr)
        raise RuntimeError(f"ogr2ogr failed for {schema}.{table} (exit={proc.returncode})")

In [19]:
conn.rollback()  # harmless; clears any aborted txn state
export_table_to_geoparquet("exports", "cel_grid_15k", OUT_DIR / "cel_grid_15k.parquet")

[INFO] Removing existing file: c:\Users\Zachary\phd_classes\uganda\data\processed\geoparquets\cel_grid_15k.parquet
[INFO] Exporting exports.cel_grid_15k -> c:\Users\Zachary\phd_classes\uganda\data\processed\geoparquets\cel_grid_15k.parquet


In [21]:
# ----------------------------
# Export all standardized tables
# ----------------------------
require_ogr2ogr()

for _src, standardized in RENAME_TABLES.items():
    if _src in skipped:
        print(f"[SKIP] {standardized} (source was missing)")
        continue
    out_fp = OUT_DIR / f"{standardized}.parquet"
    export_table_to_geoparquet(EXPORT_SCHEMA, standardized, out_fp)

print("[DONE] GeoParquet exports complete.")

[INFO] Removing existing file: c:\Users\Zachary\phd_classes\uganda\data\processed\geoparquets\cel_grid_15k.parquet
[INFO] Exporting exports.cel_grid_15k -> c:\Users\Zachary\phd_classes\uganda\data\processed\geoparquets\cel_grid_15k.parquet
[INFO] Removing existing file: c:\Users\Zachary\phd_classes\uganda\data\processed\geoparquets\cel_grid_62k.parquet
[INFO] Exporting exports.cel_grid_62k -> c:\Users\Zachary\phd_classes\uganda\data\processed\geoparquets\cel_grid_62k.parquet
[INFO] Removing existing file: c:\Users\Zachary\phd_classes\uganda\data\processed\geoparquets\cel_grid_250k.parquet
[INFO] Exporting exports.cel_grid_250k -> c:\Users\Zachary\phd_classes\uganda\data\processed\geoparquets\cel_grid_250k.parquet
[INFO] Removing existing file: c:\Users\Zachary\phd_classes\uganda\data\processed\geoparquets\cel_hex_15k.parquet
[INFO] Exporting exports.cel_hex_15k -> c:\Users\Zachary\phd_classes\uganda\data\processed\geoparquets\cel_hex_15k.parquet
[INFO] Removing existing file: c:\Users\

In [ ]:
# ----------------------------
# Summaries (optional)
# ----------------------------
for src, dst in RENAME_TABLES.items():
    if src not in skipped:
        print(dst, summarize_export_table(EXPORT_SCHEMA, dst))

conn.close()

In [ ]:
conn.close()

In [ ]:
import geopandas as gpd
from pathlib import Path

fp = Path(r"c:\\Users\\Zachary\\phd_classes\\uganda\\data\\processed\\geoparquets\\cel_grid_62k.parquet")

gdf = gpd.read_parquet(fp)

gdf_sorted = gdf.sort_values(by="name", ascending=False)
gdf_sorted

,oid,adm2_en,adm1_en,adm0_en,name,lat,lon,dist_to_road,dist_to_market,dist_to_rivers,...,ag_y_2017_pct,ag_y_2018_pct,ag_y_2019_pct,ag_y_2020_pct,ag_y_2021_pct,ag_y_2022_pct,ag_y_2023_pct,ag_y_2024_pct,centroid_wkt,geom
28,85938,Kamwenge,Western,Uganda,Rwamwanja,0.330597,30.653024,546.107180,89473.107589,5458.197094,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,POINT(238771.13860181713 36571.7594150362),"POLYGON ((238896.139 36446.759, 238896.139 366..."
716313,105713,Kamwenge,Western,Uganda,Rwamwanja,0.310268,30.700171,541.914243,91354.582477,4227.317763,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,POINT(244021.13860181713 34321.7594150362),"POLYGON ((244146.139 34196.759, 244146.139 344..."
836288,101783,Kamwenge,Western,Uganda,Rwamwanja,0.301226,30.691193,578.534202,92398.323092,3668.329174,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,POINT(243021.13860181713 33321.7594150362),"POLYGON ((243146.139 33196.759, 243146.139 334..."
256650,83206,Kamwenge,Western,Uganda,Rwamwanja,0.289917,30.646299,309.607950,94025.745739,6960.082171,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,POINT(238021.13860181713 32071.7594150362),"POLYGON ((238146.139 31946.759, 238146.139 321..."
256651,91415,Kamwenge,Western,Uganda,Rwamwanja,0.260542,30.666508,13.844246,97062.748488,3034.840928,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,POINT(240271.13860181713 28821.7594150362),"POLYGON ((240396.139 28696.759, 240396.139 289..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
963937,31559,NaN,NaN,NaN,NaN,-0.980035,30.488838,1802.639710,55353.155712,8328.068141,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,POINT(220521.13860181713 -108428.2405849638),"POLYGON ((220646.139 -108553.241, 220646.139 -..."
963938,116851,Isingiro,Western,Uganda,NaN,-0.659281,30.724749,17.075158,23142.310738,3629.618061,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,POINT(246771.13860181713 -72928.24058496379),"POLYGON ((246896.139 -73053.241, 246896.139 -7..."
963939,78220,Isingiro,Western,Uganda,NaN,-1.041159,30.634717,444.381883,45121.737628,2746.802568,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,POINT(236771.13860181713 -115178.2405849638),"POLYGON ((236896.139 -115303.241, 236896.139 -..."
963940,98371,Isingiro,Western,Uganda,NaN,-1.005036,30.684137,94.985726,38332.766318,1904.883711,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,POINT(242271.13860181713 -111178.2405849638),"POLYGON ((242396.139 -111303.241, 242396.139 -..."


In [ ]:
cols = list(gdf.columns)
print(cols)

['oid', 'adm2_en', 'adm1_en', 'adm0_en', 'name', 'lat', 'lon', 'dist_to_road', 'dist_to_market', 'dist_to_rivers', 'dist_to_rivers_and_streams', 'dist_to_rivers_plus', 'dist_to_pop_center_1', 'dist_to_pop_center_2', 'dist_to_pop_center_3', 'dist_to_road_geodesic', 'dist_to_market_geodesic', 'dist_to_rivers_geodesic', 'dist_to_rivers_and_streams_geodesic', 'dist_to_rivers_plus_geodesic', 'dist_to_pop_center_1_geodesic', 'dist_to_pop_center_2_geodesic', 'dist_to_pop_center_3_geodesic', 'dist_to_road_diff', 'dist_to_market_diff', 'dist_to_rivers_diff', 'dist_to_rivers_and_streams_diff', 'dist_to_rivers_plus_diff', 'dist_to_pop_center_1_diff', 'dist_to_pop_center_2_diff', 'dist_to_pop_center_3_diff', 'elev_avg', 'elev_sd', 'slope_avg', 'ag_total_area', 'ag_y_2017_pct', 'ag_y_2018_pct', 'ag_y_2019_pct', 'ag_y_2020_pct', 'ag_y_2021_pct', 'ag_y_2022_pct', 'ag_y_2023_pct', 'ag_y_2024_pct', 'centroid_wkt', 'geom']
